# EXP0001 — Kaggle training session (thin caller)

All logic lives in `17_Automation/` (`training`, `config_validation`, `kaggle_sync`); every parameter lives in `06_Experiments/EXP0001/config.yaml`. Do not add training code or hyperparameters here. Runbook: `06_Experiments/EXP0001/KAGGLE_RUNBOOK.md`.

**Notebook settings:** Accelerator `GPU T4 x2`, Internet `On`, input dataset `isthisdeception/ds0001-pp0001` attached.

| Session | Run cells |
|---------|-----------|
| A — LR Finder | 0 Setup → A |
| B — Training | 0 Setup → B (interrupt once, re-run) → C publish → D push |
| Resume in a new session | 0 Setup → R → B → C → D |

## 0. Setup (every session)

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/isthisdeception/thesis.git'
BRANCH = 'main'
REPO = Path('/kaggle/working/thesis')
EXP = 'EXP0001'
PLAN = 'kaggle_sync/sync_plan.example.yaml'
AUTOMATION = REPO / '17_Automation'
CONFIG = REPO / '06_Experiments' / EXP / 'config.yaml'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Optional secrets (Add-ons -> Secrets). Needed only for C (KAGGLE_USERNAME, KAGGLE_KEY) and D (GITHUB_TOKEN).
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GITHUB_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
            print(_name, 'loaded')
        except Exception:
            print(_name, 'not set')
except ImportError:
    print('kaggle_secrets unavailable (not on Kaggle)')

In [ ]:
# Pull latest main, install environment/kaggle-requirements.txt, verify attached datasets.
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} bootstrap

In [ ]:
# Environment smoke check (environment/versions.lock.md). Paste this output into observations if anything differs.
import torch, fastai, fastcore, numpy
from importlib.metadata import version
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
print('fastai', fastai.__version__, 'fastcore', fastcore.__version__, 'numpy', numpy.__version__)
print('plum-dispatch', version('plum-dispatch'))
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'GPU not visible: set Accelerator to GPU T4 x2'

In [ ]:
sys.path.insert(0, str(AUTOMATION))
from config_validation import validate_config_file
from training import find_lr, run_experiment

validate_config_file(CONFIG).raise_if_failed()
print('config PASS:', CONFIG)

## A. LR Finder (Session A only)

Report the printed `suggested_lr`; it is written into `config.yaml` (with a fresh `git_commit` / `timestamp`) before Session B.

In [ ]:
lr = find_lr(CONFIG)
print('suggested_lr =', lr['suggested_lr'])

## R. Resume from the artifact tier (only in a new session after C)

Attach `isthisdeception/thesis-experiment-artifacts` as an input first.

In [ ]:
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} resume-checkpoints

## B. Train, evaluate, export

Resumes automatically if recovery checkpoints exist. **Interruption check (once):** after the log shows `checkpoint saved EXP0001_epoch_002.pth`, press Interrupt (■), then run this cell again; the log must show `resumed from EXP0001_epoch_00N.pth`.

In [ ]:
summary = run_experiment(CONFIG)
print('resumed_from_epoch:', summary['resumed_from_epoch'], '| sessions:', summary['sessions'])
print('best_epoch:', summary['best_epoch'], '| early_stopped:', summary['early_stopped'])
print({k: round(summary['metrics']['test'][k], 4) for k in ('roc_auc', 'pr_auc', 'balanced_accuracy', 'f1', 'mcc', 'ece')})

## C. Publish checkpoints + exports to the artifact tier (needs KAGGLE_USERNAME / KAGGLE_KEY)

In [ ]:
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} publish-artifacts

## D. Push logs / metrics / manifests to GitHub (needs GITHUB_TOKEN)

Pushes to a `kaggle/` branch; the results reach `main` through a reviewed PR.

In [ ]:
from datetime import datetime
PUSH_BRANCH = f"kaggle/{EXP.lower()}-{datetime.now():%Y%m%d-%H%M}"
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} push-metadata --branch {PUSH_BRANCH} --commit-message "sync: {EXP} results from Kaggle"